In [ ]:
import os
import torch
from diffusers import UNet2DModel, AutoencoderKL, DDPMScheduler
# Use GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
SNAPSHOT_PATH = "output/snapshots"
os.makedirs(SNAPSHOT_PATH, exist_ok=True)

In [ ]:
# 1. Load Pre-trained VAE (The "Compressor")
# We use standard Stable Diffusion VAE weights to compress 512px -> 64px latents
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
vae.requires_grad_(False) # Freeze VAE
print()

In [ ]:
# Export ONLY the Decoder part of the VAE
print("Exporting VAE Decoder...")

# 1. Switch to eval mode
vae.eval()

# 2. Create dummy input matching the Latent Shape
# Shape: (Batch=1, Channels=4, Height=64, Width=64)
dummy_latent_input = torch.randn(1, 4, 64, 64).to(device)

# 3. Define output path
vae_onnx_path = f"{SNAPSHOT_PATH}/vae_decoder.onnx"

# 4. Export
torch.onnx.export(
    vae.decoder,                
    dummy_latent_input,
    vae_onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["latent_sample"],
    output_names=["sample"],      
    dynamic_axes={
        "latent_sample": {0: "batch_size"},
        "sample": {0: "batch_size"}
    },
)
print(f"VAE Decoder exported to {vae_onnx_path}")